Libraries

In [35]:
#Importar librerias
import numpy as np
import pandas as pd
import os
import csv
import matplotlib
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import ptitprince as pt
import pyodbc
import openpyxl
import pickle
import itertools
pd.options.display.max_rows = 999

In [36]:
from src.model import Model
from src.simulation import simulate_patients
from src.utils import get_labelled_sequences

data = simulate_patients(
    freq=5,      # sampling frequency of the time series, in minutes
    length=84,   # length of the time series, in days
    num=100,     # number of time series
)

# reshape the dataset from long to wide
data = data.pivot(index='ts', columns=['id'], values=['gl'])
data.columns = data.columns.get_level_values(level='id')

Importing Data

In [37]:
#carga y procesamiento de los datos
f = r"C:/Users/Anderson Mosquera/universidadean.edu.co/MAIRA ALEJANDRA GARCIA JARAMILLO - ML_Analitica_Diabetes/REPLACE-BG Dataset-79f6bdc8-3c51-4736-a39f-c4c0f71d45e5/CGM_Editada3.txt"

# 'C:/Users/Anderson Mosquera/universidadean.edu.co/MAIRA ALEJANDRA GARCIA JARAMILLO - ML_Analitica_Diabetes/REPLACE-BG Dataset-79f6bdc8-3c51-4736-a39f-c4c0f71d45e5/Data Tables/HDeviceCGM.txt'
MasterDF = pd.read_csv(f, sep='|', header=0, low_memory = False)
MasterDF = MasterDF[MasterDF.PtID.isin([183, 184,  14, 220, 233,  62,  17, 186,  52, 216, 115,  37, 244])]
MasterDF = MasterDF[MasterDF['RecordType'] == 'CGM']

Feature Engineering

In [38]:
#Creating a date time column
MasterDF['Today'] = datetime.today().date()
MasterDF['Date'] = MasterDF['Today'] + pd.to_timedelta(MasterDF['DeviceDtTmDaysFromEnroll'], unit='d')
MasterDF['DeviceTm'] = MasterDF.DeviceTm.astype('str')
MasterDF['DeviceTm'] = MasterDF['DeviceTm'].str[:-2]+ '00'
MasterDF['DateTime'] = MasterDF.Date.astype('str')+ ' '+ MasterDF.DeviceTm
MasterDF['DateTime'] = pd.to_datetime(MasterDF.DateTime, format='%Y-%m-%d %H:%M:%S')
#selecting just the columns for Giammarino's code to run
MasterDF = MasterDF[['PtID','DateTime','GlucoseValue']]
MasterDF = MasterDF.rename(columns={'DateTime':'ts','PtID':'id','GlucoseValue':'gl'})
MasterDF= MasterDF.reset_index(drop=True)
MasterDF = MasterDF.drop_duplicates(subset=['ts','id'])

In [39]:
MasterDF['Dia_Noche'] = MasterDF['ts'].dt.hour.between(7, 18, inclusive='both') \
    .replace({True: 'Dia', False: 'Noche'})

In [40]:
MasterDF = MasterDF[MasterDF['Dia_Noche']=='Noche']
MasterDF=MasterDF[['ts', 'id','gl']]

In [41]:
data = MasterDF

In [42]:
# reshape the dataset from long to wide
data = data.pivot(index='ts', columns=['id'], values=['gl'])
data.columns = data.columns.get_level_values(level='id')

In [43]:
# Run Giammarino's code
from src.model import Model
from src.simulation import simulate_patients
from src.utils import get_labelled_sequences

# minimum percentage of time that the patient must have worn the device over a given week
time_worn_threshold = 0.7

# glucose threshold below which we detect the onset of hypoglycemia, in mg/dL
glucose_threshold = 54

# minimum length of a hypoglycemic event, in minutes
event_duration_threshold = 15


# split the dataset into sequences
sequences = get_labelled_sequences(
    data=data,
    time_worn_threshold=time_worn_threshold,
    glucose_threshold=glucose_threshold,
    event_duration_threshold=event_duration_threshold,
)

In [44]:
from sklearn.model_selection import StratifiedKFold
# split the sequences into folds
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

# create a list for storing the results for each fold
results = []

# loop across the folds
for i, (train_index, test_index) in enumerate(skf.split(X=[s['X'] for s in sequences], y=[s['Y'] for s in sequences])):
    
    # fit the model to the training set
    model = Model()

    model.fit(
        sequences=[sequences[i] for i in train_index],
        sequence_length=int(7 * 24 * 60 // 5),
        l1_penalty=0.0005,
        l2_penalty=0.01,
        learning_rate=0.0001,
        batch_size=52,
        epochs=1000,
        seed=42,
        verbose=0
    )

    # evaluate the model on the test set
    metrics = model.evaluate(sequences=[sequences[i] for i in test_index])

    # save the results
    results.append(metrics)

# organize the results in a data frame
results = pd.DataFrame(results)

# average the results
print(results.mean())

ValueError: Found array with 0 sample(s) (shape=(0,)) while a minimum of 1 is required.